In [ ]:
from pyspark.sql import SparkSession
import pandas as pd

# create spark session
spark = (SparkSession.builder.appName("Cancer Database").getOrCreate())

In [3]:
db = spark.sparkContext.textFile('project3_data.csv')
db

project3_data.csv MapPartitionsRDD[1] at textFile at DirectMethodHandleAccessor.java:104

In [4]:
data_path = 'project3_data.csv'

df = spark.read.csv(
    data_path,
    header=True,
    inferSchema=True
)

df.printSchema()

train, test = df.randomSplit([0.8, 0.2], seed=42)

root
 |-- id: integer (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- Radius_mean: double (nullable = true)
 |-- Texture_mean: double (nullable = true)
 |-- perimeter_mean: double (nullable = true)
 |-- area_mean: double (nullable = true)
 |-- smoothness_mean: double (nullable = true)
 |-- compactness_mean: double (nullable = true)
 |-- concavity_mean: double (nullable = true)
 |-- concave points_mean: double (nullable = true)
 |-- symmetry_mean: double (nullable = true)
 |-- fractal_dimension_mean: double (nullable = true)
 |-- radius_se: double (nullable = true)
 |-- texture_se: double (nullable = true)
 |-- perimeter_se: double (nullable = true)
 |-- area_se: double (nullable = true)
 |-- smoothness_se: double (nullable = true)
 |-- compactness_se: double (nullable = true)
 |-- concavity_se: double (nullable = true)
 |-- concave points_se: double (nullable = true)
 |-- symmetry_se: double (nullable = true)
 |-- fractal_dimension_se: double (nullable = true)
 |-- radi

In [ ]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler

label_indexer = StringIndexer(
    inputCol="diagnosis",
    outputCol="label",
    handleInvalid="keep"
)

label_col = "diagnosis" 

numeric_features = [
    col for col, dtype in df.dtypes
    if dtype in ("int", "double") and col != label_col
]

assembler = VectorAssembler(
    inputCols=numeric_features,
    outputCol="features"
)


In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

rf = RandomForestClassifier(
    labelCol='label',
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

pipeline = Pipeline(stages=[
    label_indexer, 
    assembler,
    rf
])

model = pipeline.fit(train)
rf_output = model.transform(test)

25/12/21 18:43:36 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [7]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
)

rf_f1 = evaluator.evaluate(rf_output, {evaluator.metricName : "f1"})
rf_precision = evaluator.evaluate(rf_output, {evaluator.metricName : "precisionByLabel"})
rf_recall = evaluator.evaluate(rf_output, {evaluator.metricName : "recallByLabel"})
rf_accuracy = evaluator.evaluate(rf_output, {evaluator.metricName : "accuracy"})

print(f'F1: {rf_f1:.3f}')
print(f"Precision: {rf_precision:.3f}")
print(f"Recall: {rf_recall:.3f}")
print(f"Accuracy: {rf_accuracy:.3f}")

F1: 0.965
Precision: 0.960
Recall: 0.980
Accuracy: 0.965


In [8]:
# visualization of the trees

rf_model = model.stages[-1]

tree_splits = rf_model.toDebugString
for i in range(0,len(numeric_features)):
    f_name = ('feature '+str(i))
    if f_name in tree_splits: 
        tree_splits = tree_splits.replace(f_name, numeric_features[i])

print(tree_splits)

RandomForestClassificationModel: uid=RandomForestClassifier_d127ae8cafe4, numTrees=100, numClasses=3, numFeatures=31
  Tree 0 (weight 1.0):
    If (Texture_mean8 <= 0.14525)
     If (Texture_mean1 <= 17.0)
      If (Texture_mean8 <= 0.11705)
       If (Radius_mean <= 14.864999999999998)
        If (Texture_mean8 <= 0.10955000000000001)
         Predict: 0.0
        Else (Texture_mean8 > 0.10955000000000001)
         If (Radius_mean5 <= 0.00345)
          Predict: 1.0
         Else (Radius_mean5 > 0.00345)
          Predict: 0.0
       Else (Radius_mean > 14.864999999999998)
        If (Texture_mean3 <= 97.14)
         Predict: 1.0
        Else (Texture_mean3 > 97.14)
         Predict: 0.0
      Else (Texture_mean8 > 0.11705)
       If (Texture_mean <= 20.994999999999997)
        If (Texture_mean0 <= 0.002698)
         If (id <= 892521.0)
          Predict: 1.0
         Else (id > 892521.0)
          Predict: 0.0
        Else (Texture_mean0 > 0.002698)
         Predict: 0.0
       Else 

In [9]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

layers = [len(numeric_features), 5, 4, 3]

mp = MultilayerPerceptronClassifier(maxIter=200, layers=layers, blockSize=128, seed=1234, labelCol='label', featuresCol='features',)

mp_pipeline = Pipeline(stages=[
    label_indexer, 
    assembler,
    mp
])

mp_model = mp_pipeline.fit(train)
mp_output = model.transform(test)

25/12/21 18:43:42 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/12/21 18:43:42 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
25/12/21 18:43:42 WARN InstanceBuilder$JavaBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


In [10]:
# model 2 eval
mp_f1 = evaluator.evaluate(mp_output, {evaluator.metricName : "f1"})
mp_precision = evaluator.evaluate(mp_output, {evaluator.metricName : "precisionByLabel"})
mp_recall = evaluator.evaluate(mp_output, {evaluator.metricName : "recallByLabel"})
mp_accuracy = evaluator.evaluate(mp_output, {evaluator.metricName : "accuracy"})

print(f'F1: {mp_f1:.3f}')
print(f"Precision: {mp_precision:.3f}")
print(f"Recall: {mp_recall:.3f}")
print(f"Accuracy: {mp_accuracy:.3f}")

F1: 0.965
Precision: 0.960
Recall: 0.980
Accuracy: 0.965


In [11]:
# end session
spark.stop()